# 🚀 AuraNet Training — Kaggle GPU/TPU

Train AuraNet speech enhancement model using Kaggle's free accelerators.

**Setup:**
1. Enable GPU/TPU: Settings → Accelerator → GPU T4 x2 (or TPU)
2. Enable Internet: Settings → Internet → On
3. Run All

---

In [ ]:
# =============================================================================
# KAGGLE ENVIRONMENT SETUP — Load from uploaded Dataset
# =============================================================================

import os, sys, shutil, zipfile

PROJECT_ROOT = "/kaggle/working/auranet"
KAGGLE_INPUT = "/kaggle/input"

# Find the auranet project in attached datasets
source_found = False
for ds in os.listdir(KAGGLE_INPUT):
    ds_path = os.path.join(KAGGLE_INPUT, ds)
    if not os.path.isdir(ds_path):
        continue

    # Check for zip file inside the dataset
    for f in os.listdir(ds_path):
        fpath = os.path.join(ds_path, f)
        if f.endswith('.zip'):
            print(f"📦 Found zip: {ds}/{f}")
            if os.path.exists(PROJECT_ROOT):
                shutil.rmtree(PROJECT_ROOT)
            os.makedirs(PROJECT_ROOT, exist_ok=True)
            with zipfile.ZipFile(fpath, 'r') as zf:
                zf.extractall(PROJECT_ROOT)
            source_found = True
            break

    # Check for direct project files (config.yaml or model.py = auranet project)
    if not source_found:
        if os.path.exists(os.path.join(ds_path, "config.yaml")) or \
           os.path.exists(os.path.join(ds_path, "model.py")):
            print(f"📂 Found project files in dataset: {ds}")
            if os.path.exists(PROJECT_ROOT):
                shutil.rmtree(PROJECT_ROOT)
            shutil.copytree(ds_path, PROJECT_ROOT)
            source_found = True

    if source_found:
        break

if not source_found:
    available = os.listdir(KAGGLE_INPUT) if os.path.exists(KAGGLE_INPUT) else []
    print("❌ AuraNet project not found in attached datasets!")
    print(f"   Available datasets: {available}")
    print("   Make sure you added your dataset: Add Data → Your Datasets")
    raise FileNotFoundError("Attach your auranet dataset to this notebook")

# Handle nested folder (zip might have a single root folder)
items = os.listdir(PROJECT_ROOT)
if len(items) == 1 and os.path.isdir(os.path.join(PROJECT_ROOT, items[0])):
    nested = os.path.join(PROJECT_ROOT, items[0])
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), os.path.join(PROJECT_ROOT, item))
    os.rmdir(nested)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Paths
DATA_DIR = os.path.join(PROJECT_ROOT, "datasets")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
OUTPUT_DIR = "/kaggle/working/outputs"

for d in [DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

IN_COLAB = False
IN_KAGGLE = True

# Install dependencies
!pip install -q librosa soundfile pyyaml scipy

print(f"\n✅ Project: {PROJECT_ROOT}")
print(f"📊 Data Dir: {DATA_DIR}")
print(f"💾 Checkpoint Dir: {CHECKPOINT_DIR}")
py_files = [f for f in os.listdir(PROJECT_ROOT) if f.endswith('.py')]
print(f"📁 Files: {len(py_files)} .py files")
print(f"   {', '.join(sorted(py_files)[:8])}...")

In [ ]:
# =============================================================================
# GPU SETUP
# =============================================================================

import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  No GPU! Go to Settings → Accelerator → GPU T4 x2")

print(f"✅ Device: {DEVICE}")

In [ ]:
# =============================================================================
# IMPORTS & CONFIGURATION
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import yaml

from model import AuraNet, create_auranet
from dataset import AuraNetDataset
from loss import AuraNetLoss
from utils.stft import CausalSTFT
print("✅ AuraNet modules imported")

config_path = os.path.join(PROJECT_ROOT, "config.yaml")
if os.path.exists(config_path):
    with open(config_path) as f:
        CONFIG = yaml.safe_load(f)
    print("✅ Config loaded")
else:
    CONFIG = {}
    print("⚠️  No config.yaml found, using defaults")

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

BATCH_SIZE = 16
NUM_EPOCHS = 25
USE_SYNTHETIC = False  # Use real data (LibriSpeech)

print(f"📋 Training Config:")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Data: {'Synthetic' if USE_SYNTHETIC else 'Real (LibriSpeech)'}")

In [ ]:
# =============================================================================
# DOWNLOAD DATASET FROM INTERNET
# =============================================================================
# Downloads LibriSpeech (clean speech) + generates noise samples.
# Requires: Settings → Internet → ON
# Estimated download: ~6 GB, processing: ~15 min

import torchaudio
import torch
import random
import subprocess
from pathlib import Path

speech_dir = Path(DATA_DIR) / "speech"
noise_dir = Path(DATA_DIR) / "noise"

def check_dataset_exists(min_files=100):
    if not speech_dir.exists():
        return False
    files = list(speech_dir.glob("*.wav")) + list(speech_dir.glob("*.flac"))
    return len(files) >= min_files

def check_internet():
    """Verify internet access is enabled."""
    try:
        import urllib.request
        urllib.request.urlopen("https://www.openslr.org", timeout=10)
        return True
    except Exception:
        return False

def download_librispeech_wget(num_hours=40):
    """Download LibriSpeech using wget (faster and more reliable on Kaggle)."""
    import tarfile

    speech_dir.mkdir(parents=True, exist_ok=True)

    url = "https://www.openslr.org/resources/12/train-clean-100.tar.gz"
    tar_path = Path(DATA_DIR) / "train-clean-100.tar.gz"
    extract_dir = Path(DATA_DIR) / "LibriSpeech"

    # Download
    if not tar_path.exists() and not extract_dir.exists():
        print("📥 Downloading LibriSpeech train-clean-100 (~6 GB)...")
        print(f"   URL: {url}")
        result = subprocess.run(
            ["wget", "-q", "--show-progress", "-O", str(tar_path), url],
            capture_output=False
        )
        if result.returncode != 0:
            print("⚠️  wget failed, trying torchaudio fallback...")
            return download_librispeech_torchaudio(num_hours)

    # Extract
    if not extract_dir.exists():
        print("📦 Extracting archive...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=str(DATA_DIR))
        tar_path.unlink()  # Remove tar to save space
        print("✅ Extracted")

    # Convert FLAC to WAV at 16kHz
    libri_root = Path(DATA_DIR) / "LibriSpeech" / "train-clean-100"
    flac_files = list(libri_root.rglob("*.flac"))
    random.shuffle(flac_files)

    total_duration = 0
    target_duration = num_hours * 3600

    print(f"🔄 Converting to 16kHz WAV (~{num_hours} hours)...")
    for i, flac_path in enumerate(tqdm(flac_files)):
        if total_duration >= target_duration:
            break
        try:
            waveform, sr = torchaudio.load(str(flac_path))
            duration = waveform.shape[1] / sr
            total_duration += duration

            if sr != 16000:
                waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

            out_path = speech_dir / f"speech_{i:05d}.wav"
            torchaudio.save(str(out_path), waveform, 16000)
        except Exception as e:
            print(f"   Skipping {flac_path.name}: {e}")
            continue

    print(f"✅ Saved {total_duration/3600:.1f} hours of clean speech ({i+1} files)")
    return True

def download_librispeech_torchaudio(num_hours=40):
    """Fallback: download via torchaudio API."""
    speech_dir.mkdir(parents=True, exist_ok=True)

    print("📥 Downloading via torchaudio API...")
    try:
        dataset = torchaudio.datasets.LIBRISPEECH(
            root=str(DATA_DIR),
            url="train-clean-100",
            download=True
        )

        indices = list(range(len(dataset)))
        random.shuffle(indices)

        total_duration = 0
        target_duration = num_hours * 3600

        print(f"📊 Selecting ~{num_hours} hours of speech...")
        for i, idx in enumerate(tqdm(indices)):
            if total_duration >= target_duration:
                break
            waveform, sample_rate, *_ = dataset[idx]
            duration = waveform.shape[1] / sample_rate
            total_duration += duration
            if sample_rate != 16000:
                waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
            output_path = speech_dir / f"speech_{i:05d}.wav"
            torchaudio.save(str(output_path), waveform, 16000)

        print(f"✅ Saved {total_duration/3600:.1f} hours of clean speech")
        return True
    except Exception as e:
        print(f"❌ LibriSpeech download failed: {e}")
        return False

def generate_noise(noise_dir, num_samples=500, duration=10.0):
    """Generate synthetic noise samples (white, pink, brown, ambient)."""
    noise_dir.mkdir(parents=True, exist_ok=True)
    sample_rate = 16000
    num_samples_audio = int(duration * sample_rate)

    for i in tqdm(range(num_samples), desc="Generating noise"):
        noise_type = random.choice(['white', 'pink', 'brown', 'ambient'])
        if noise_type == 'white':
            noise = torch.randn(1, num_samples_audio) * 0.3
        elif noise_type == 'pink':
            white = torch.randn(num_samples_audio)
            b = [0.049922035, -0.095993537, 0.050612699, -0.004408786]
            a = [1, -2.494956002, 2.017265875, -0.522189400]
            from scipy.signal import lfilter
            noise = torch.tensor(lfilter(b, a, white.numpy())).unsqueeze(0).float() * 0.3
        elif noise_type == 'brown':
            white = torch.randn(num_samples_audio)
            noise = torch.cumsum(white, dim=0).unsqueeze(0)
            noise = noise / noise.abs().max() * 0.3
        else:
            noise = torch.randn(1, num_samples_audio) * 0.1
            t = torch.linspace(0, duration, num_samples_audio)
            noise += 0.1 * torch.sin(2 * 3.14159 * random.uniform(20, 100) * t).unsqueeze(0)
        output_path = noise_dir / f"noise_{i:04d}.wav"
        torchaudio.save(str(output_path), noise, sample_rate)
    print(f"✅ Generated {num_samples} noise samples")

# ---- Run ----
if check_dataset_exists():
    print("✅ Dataset already exists")
    print(f"   Speech: {len(list(speech_dir.glob('*.wav')))} files")
    print(f"   Noise: {len(list(noise_dir.glob('*.wav')))} files")
else:
    if not check_internet():
        print("❌ No internet access!")
        print("   Go to: Settings → Internet → Turn ON")
        print("   Then re-run this cell")
    else:
        print("🌐 Internet: OK")
        success = download_librispeech_wget(num_hours=40)
        if success:
            print("\n🔊 Generating noise samples...")
            generate_noise(noise_dir, num_samples=500)
            print(f"\n✅ Dataset ready!")
            print(f"   Speech: {len(list(speech_dir.glob('*.wav')))} files")
            print(f"   Noise: {len(list(noise_dir.glob('*.wav')))} files")
        else:
            print("\n⚠️  Falling back to synthetic data (training cell will handle this)")

In [ ]:
# =============================================================================
# CREATE MODEL & DATASET
# =============================================================================

model = create_auranet(CONFIG)
model = model.to(DEVICE)
print(f"\n🧠 Model created: {model.count_parameters():,} parameters")

dataset = AuraNetDataset(
    clean_dir=str(speech_dir),
    noise_dir=str(noise_dir),
    synthetic_mode=False,
)

if len(dataset) == 0:
    print("⚠️ Using synthetic data")
    dataset = AuraNetDataset(
        synthetic_mode=True,
        num_synthetic_samples=100,
    )

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

print(f"📊 Dataset: {len(dataset)} samples, {len(dataloader)} batches")

In [ ]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = AuraNetLoss()
stft = CausalSTFT().to(DEVICE)

# Resume from checkpoint if exists
start_epoch = 0
best_loss = float('inf')
checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest.pt")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint.get('best_loss', float('inf'))
    print(f"✅ Resumed from epoch {start_epoch}")

print(f"\n🚀 Starting training from epoch {start_epoch + 1}...")
print(f"   Target: {NUM_EPOCHS} epochs")
print(f"   Device: {DEVICE}")
print()

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    epoch_loss = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        noisy_stft = batch['noisy_stft'].to(DEVICE)
        clean_stft = batch['clean_stft'].to(DEVICE)
        clean_audio = batch['clean_audio'].to(DEVICE)

        optimizer.zero_grad()

        enhanced_stft, _, _ = model(noisy_stft)
        enhanced_audio = stft.inverse(enhanced_stft)

        loss, _ = criterion(enhanced_stft, clean_stft, enhanced_audio, clean_audio)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

    is_best = avg_loss < best_loss
    if is_best:
        best_loss = avg_loss

    # Save checkpoint
    ckpt_dict = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_loss': best_loss,
    }
    torch.save(ckpt_dict, checkpoint_path)

    if is_best:
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print(f"💾 Best model saved!")

    # Also save to /kaggle/working for easy download
    if is_best:
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")

print(f"\n✅ Training complete! Best loss: {best_loss:.4f}")

In [ ]:
# =============================================================================
# INFERENCE TEST
# =============================================================================

import torchaudio
from IPython.display import Audio, display

best_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print("✅ Loaded best model")

model.eval()

# Test with synthetic audio
sample_rate = 16000
duration = 2.0
t = torch.linspace(0, duration, int(sample_rate * duration))

clean = sum((1/h) * torch.sin(2 * 3.14159 * 150 * h * t) for h in range(1, 6))
clean = clean / clean.abs().max() * 0.5
noise = torch.randn_like(t) * 0.15
noisy = clean + noise

with torch.no_grad():
    audio_in = noisy.unsqueeze(0).to(DEVICE)
    noisy_stft = stft(audio_in)
    enhanced_stft, _, _ = model(noisy_stft)
    enhanced = stft.inverse(enhanced_stft).cpu().squeeze()

print("🔊 Noisy:")
display(Audio(noisy.numpy(), rate=sample_rate))
print("🔊 Enhanced:")
display(Audio(enhanced.numpy(), rate=sample_rate))
print("🔊 Clean:")
display(Audio(clean.numpy(), rate=sample_rate))

In [ ]:
# =============================================================================
# SAVE OUTPUT FOR DOWNLOAD
# =============================================================================

import shutil

output_dir = "/kaggle/working"

# Copy best model to output root
best_src = os.path.join(CHECKPOINT_DIR, "best_model.pt")
latest_src = os.path.join(CHECKPOINT_DIR, "latest.pt")

if os.path.exists(best_src):
    shutil.copy2(best_src, os.path.join(output_dir, "best_model.pt"))
    size_mb = os.path.getsize(best_src) / 1e6
    print(f"✅ best_model.pt → Output tab ({size_mb:.1f} MB)")

if os.path.exists(latest_src):
    shutil.copy2(latest_src, os.path.join(output_dir, "latest.pt"))
    print(f"✅ latest.pt → Output tab")

print(f"\n📥 To download: Click 'Output' tab on the right → Download files")
print(f"   Then place best_model.pt in your local auranet/checkpoints/ folder")

---

## 📥 After Training

1. Click the **Output** tab on the right side panel
2. Download `best_model.pt`
3. Place it in your local repo: `auranet/checkpoints/best_model.pt`
4. In VS Code, run: `python scripts/load_trained_model.py --model checkpoints/best_model.pt`